# Buat liat distribusi data yang di crop berdasarkan nilai min count yang dipakai

In [1]:
!pip install gensim top2vec


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from top2vec import Top2Vec
import pandas as pd
from bertopic import BERTopic
from bertopic.backend import MultiModalBackend
from bertopic.representation import KeyBERTInspired
import re
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import random
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

c:\Users\Allen\Documents\Python Env\environments\derp_learning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv(r"C:\Users\Allen\Documents\GitHub\thesis-research\combined_tweets_dataset\sample_v2\vibe_coding_combined_translated.csv")

In [4]:
df = df[["full_text_translated", "image_url"]]

In [5]:
df.head()

,full_text_translated,image_url
0,@karpathy me vibe coding and hope it can just ...,https://pbs.twimg.com/media/Gi0a82HaQAA97Q-.jpg
1,vibes capital meets vibe coding = greatest eco...,NaN
2,vibe coding https://t.co/1hHVMmunrw,https://pbs.twimg.com/media/Gi0f9fAWgAAgox6.jpg
3,can attest. vibe coding is hella fun and borde...,NaN
4,I have built a few app ideas in my spare time ...,NaN


In [6]:
def preprocess_tweet(text):
    #lower case text
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove user mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags
    text = re.sub(r'#\w+', '', text)
    #remove symbols
    text = re.sub(r'[^\w\s]', '', text)
    #remove numbers
    text = re.sub(r'\d+', '', text)
    #Replace all "vibe code" into vibecode
    text = re.sub(r'vibe code', 'vibecode', text)
    #Replace all "vibe coding" into vibecoding
    text = re.sub(r'vibe coding', 'vibecoding', text)
    #Relace all "vibe coded" into vibecoded
    text = re.sub(r'vibe coded', 'vibecoded', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [7]:
df_preprocessed = df['full_text_translated'].fillna('').apply(preprocess_tweet)
df_preprocessed = pd.concat([df_preprocessed, df['image_url']], axis=1)

In [8]:
df_preprocessed = df_preprocessed[df_preprocessed['full_text_translated'].apply(lambda x: len(x.split()) >= 4)]
df_preprocessed.count()

full_text_translated    20511
image_url                6495
dtype: int64

In [12]:
from collections import Counter
import pandas as pd

# Asumsi df['text'] adalah tweet kamu
all_words = ' '.join(df_preprocessed['full_text_translated']).split()
word_freq = Counter(all_words)

# Lihat berapa banyak kata yang muncul hanya 1-5 kali
freq_df = pd.DataFrame(word_freq.items(), columns=['word', 'count'])
freq_df.head()


,word,count
0,me,1640
1,vibecoding,17566
2,and,11496
3,hope,77
4,it,6401


In [16]:
print(freq_df[freq_df['count'] < 10].shape[0], "kata akan terbuang jika min_count=10")

19873 kata akan terbuang jika min_count=10
